<a href="https://colab.research.google.com/github/Rumas0/Thesis_work_SSL-imbalance/blob/main/ResNet_Augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import shutil
import pandas as pd

## Mount Drive
from google.colab import drive
drive.mount('/content/drive')

##Create directories
os.makedirs('data/isic2019', exist_ok=True)
os.makedirs('data/labeled_real', exist_ok=True)

##Extract ZIP from Drive
zip_path = '/content/drive/MyDrive/Thesis-work/ISIC_2019_Training_Input.zip'
print("Extracting... (takes 5-10 minutes)")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('data/isic2019/')
print("✓ Extraction complete")

# 3. Filter to your 691 images
backup_dir = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
train = pd.read_csv(f'{backup_dir}/expA_train.csv')
val = pd.read_csv(f'{backup_dir}/expA_val.csv')
test = pd.read_csv(f'{backup_dir}/expA_test.csv')
all_images = pd.concat([train, val, test])['image'].unique()

source = 'data/isic2019/ISIC_2019_Training_Input/'
target = 'data/labeled_real/'

found = 0
for img_id in all_images:
    src = f'{source}/{img_id}.jpg'
    dst = f'{target}/{img_id}.jpg'
    if os.path.exists(src):
        shutil.copy(src, dst)
        found += 1

print(f"✓ Copied {found}/{len(all_images)} images to {target}")

# 4. Verify
sample = os.listdir(target)[0]
from PIL import Image
img = Image.open(f'{target}/{sample}')
print(f"Sample: {sample}, Size: {img.size}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting... (takes 5-10 minutes)
✓ Extraction complete
✓ Copied 691/691 images to data/labeled_real/
Sample: ISIC_0028391.jpg, Size: (600, 450)


#**Goal: Overcome the 3-layer CNN limitation with transfer learning**
#**ResNet18 Pretrained + Heavy Augmentation**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import Counter
import json
import os
from google.colab import drive

drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


##LOAD DATA
backup_dir = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
train_df = pd.read_csv(f'{backup_dir}/expA_train.csv')
val_df = pd.read_csv(f'{backup_dir}/expA_val.csv')
test_df = pd.read_csv(f'{backup_dir}/expA_test.csv')

IMAGE_DIR = 'data/labeled_real'

print("Class distribution (train):")
print(train_df['label'].value_counts())


## HEAVY AUGMENTATION FOR MINORITY CLASS

## Normal transform for BKL and NV
normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## Strong augmentation specifically for MEL (minority)
mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## Test transform (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class AugmentedDataset(Dataset):
    def __init__(self, df, image_dir, mel_multiplier=5, transform_normal=None, transform_mel=None):
        self.df = df
        self.image_dir = image_dir
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel

        ## Expanding MEL samples by repeating them
        mel_rows = df[df['label'] == 'MEL']
        other_rows = df[df['label'] != 'MEL']

        mel_expanded = pd.concat([mel_rows] * mel_multiplier, ignore_index=True)
        self.expanded_df = pd.concat([mel_expanded, other_rows], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

        print(f"Original: {len(df)} | Expanded: {len(self.expanded_df)}")
        print(f"Expanded distribution: {self.expanded_df['label'].value_counts().to_dict()}")

    def __len__(self):
        return len(self.expanded_df)

    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        img_path = f"{self.image_dir}/{row['image']}.jpg"
        img = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[row['label']]

        ##Using strong augmentation for MEL, normal for others
        if row['label'] == 'MEL' and self.transform_mel:
            img = self.transform_mel(img)
        elif self.transform_normal:
            img = self.transform_normal(img)
        else:
            img = test_transform(img)

        return img, label

## Creating datasets
train_ds = AugmentedDataset(train_df, IMAGE_DIR, mel_multiplier=5,
                            transform_normal=normal_transform, transform_mel=mel_transform)
val_ds = AugmentedDataset(val_df, IMAGE_DIR, mel_multiplier=1,
                          transform_normal=test_transform, transform_mel=test_transform)
test_ds = AugmentedDataset(test_df, IMAGE_DIR, mel_multiplier=1,
                           transform_normal=test_transform, transform_mel=test_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)


## RESNET18 PRETRAINED MODEL


class ResNetClassifier(nn.Module):
    def __init__(self, num_classes, freeze_early=True):
        super().__init__()
        # Load pretrained ResNet18
        resnet = models.resnet18(pretrained=True)

        # Remove final FC layer, keep feature extractor (512-dim output)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])

        # Freeze early layers (first 6 blocks) if specified
        if freeze_early:
            for param in list(self.encoder.parameters())[:30]:  # Approx first 6 blocks
                param.requires_grad = False
            print("✓ Froze early ResNet layers")

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        features = self.encoder(x)
        features = features.squeeze(-1).squeeze(-1)  ## [B, 512, 1, 1] → [B, 512]
        return self.classifier(features)

model = ResNetClassifier(len(train_ds.classes), freeze_early=True).to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")


## LOSS & OPTIMIZER (Moderate Settings)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

# Moderate weights (not too aggressive)
weight_dict = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}
weights_list = [weight_dict[cls] for cls in train_ds.classes]
weights = torch.tensor(weights_list, dtype=torch.float32).to(device)

print(f"Class order: {train_ds.classes}")
print(f"Weights: {dict(zip(train_ds.classes, weights_list))}")

criterion = FocalLoss(alpha=weights, gamma=1.5)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)


## TRAINING LOOP


def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return accuracy_score(all_labels, all_preds), all_labels, all_preds

## Train
print("\nTraining ResNet18 + Augmentation...")
epochs = 30
best_val = 0
patience = 7
epochs_no_improve = 0

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, _, _ = evaluate(model, val_loader)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), 'best_resnet18.pth')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 3 == 0:
        print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val={val_acc:.3f}")

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break


## EVALUATION


model.load_state_dict(torch.load('best_resnet18.pth'))
test_acc, true_labels, pred_labels = evaluate(model, test_loader)

print("\n" + "="*50)
print("RESULTS: RESNET18 + AUGMENTATION")
print("="*50)
print(f"Test Accuracy: {test_acc:.3f}")

report = classification_report(true_labels, pred_labels, target_names=train_ds.classes, output_dict=True)
print(classification_report(true_labels, pred_labels, target_names=train_ds.classes))

mel_recall = report['MEL']['recall']
print(f"\n>>> MEL Recall: {mel_recall:.1%} <<<")
print(f"Predictions: {Counter(pred_labels)}")

## Saving to Drive
results = {
    'model': 'ResNet18_pretrained',
    'augmentation': 'heavy_mel_5x',
    'weights': weight_dict,
    'test_accuracy': test_acc,
    'mel_recall': mel_recall,
    'report': report
}

with open('resnet18_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), '/content/drive/MyDrive/Thesis-work/thesis_backup_day1/resnet18_best.pth')
print("\n✓ Saved to Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu
Class distribution (train):
label
NV     349
BKL     99
MEL     35
Name: count, dtype: int64
Original: 483 | Expanded: 623
Expanded distribution: {'NV': 349, 'MEL': 175, 'BKL': 99}
Original: 104 | Expanded: 104
Expanded distribution: {'NV': 76, 'BKL': 21, 'MEL': 7}
Original: 104 | Expanded: 104
Expanded distribution: {'NV': 75, 'BKL': 21, 'MEL': 8}


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✓ Froze early ResNet layers
Total params: 11,242,563 | Trainable: 10,559,491
Class order: ['BKL', 'MEL', 'NV']
Weights: {'BKL': 2.0, 'MEL': 4.0, 'NV': 1.0}

Training ResNet18 + Augmentation...
Epoch 3: Train=0.886, Val=0.740
Epoch 6: Train=0.926, Val=0.702
Epoch 9: Train=0.973, Val=0.731
Epoch 12: Train=0.968, Val=0.731
Epoch 15: Train=0.965, Val=0.788
Early stopping at epoch 15

RESULTS: RESNET18 + AUGMENTATION
Test Accuracy: 0.779
              precision    recall  f1-score   support

         BKL       0.53      0.48      0.50        21
         MEL       0.40      0.50      0.44         8
          NV       0.89      0.89      0.89        75

    accuracy                           0.78       104
   macro avg       0.61      0.62      0.61       104
weighted avg       0.78      0.78      0.78       104


>>> MEL Recall: 50.0% <<<
Predictions: Counter({np.int64(2): 75, np.int64(0): 19, np.int64(1): 10})

✓ Saved to Drive
